# Recodificación: Encuesta 2026.sav

Este notebook carga el archivo SPSS, recodifica las variables clave con etiquetas de texto
y construye el diccionario de orden categórico para las funciones de análisis ponderado.

**Estructura del archivo:**
- 5.000 casos
- 587 variables
- Factores de expansión: `fe_hogar` y `fe_personas`

**Orden de ejecución:**
1. Carga del archivo
2. Diccionario de variables originales
3. Tratamiento de NS/NR numéricos
4. Renombrado de variables principales
5. Recodificaciones (texto directo, sin sufijos)
6. Eliminación de columnas originales numéricas ya reemplazadas
7. Funciones de análisis ponderado (`ORDEN_CATEGORIAS` + `dstats`)

## 1. Carga del archivo

In [16]:
import pyreadstat
import pandas as pd
import numpy as np
import unicodedata
import re
import seaborn as sns
import matplotlib.pyplot as plt

In [17]:
# Carga el archivo .sav y su metadata
df, meta = pyreadstat.read_sav("/Users/tomas/GitHub/eaui_subtel/data/sav/2026.sav")

print(f"Filas:    {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

Filas:    5000
Columnas: 587


In [ ]:
# ── EJECUTAR JUSTO DESPUÉS DE CARGAR EL ARCHIVO ──────────────
# Calcula GSE con los nombres y valores originales del SPSS
# antes de cualquier renombrado o recodificación.

def _educ_grupo_gse(e):
    if pd.isna(e): return None
    e = int(e)
    if e <= 3:   return 'basica'
    elif e <= 7: return 'media'
    elif e <= 9: return 'tecnica'
    else:        return 'universitaria'

_matriz_gse = {
    (1, 'basica'): 'E',  (1, 'media'): 'E',  (1, 'tecnica'): 'D',  (1, 'universitaria'): 'D',
    (2, 'basica'): 'E',  (2, 'media'): 'D',  (2, 'tecnica'): 'D',  (2, 'universitaria'): 'C3',
    (3, 'basica'): 'D',  (3, 'media'): 'C3', (3, 'tecnica'): 'C3', (3, 'universitaria'): 'C2',
    (4, 'basica'): 'C3', (4, 'media'): 'C2', (4, 'tecnica'): 'C2', (4, 'universitaria'): 'C1',
    (5, 'basica'): 'C2', (5, 'media'): 'C1', (5, 'tecnica'): 'C1', (5, 'universitaria'): 'AB',
    (6, 'basica'): 'C1', (6, 'media'): 'AB', (6, 'tecnica'): 'AB', (6, 'universitaria'): 'AB',
}

_orden_gse = ['E', 'D', 'C3', 'C2', 'C1', 'AB']

# Usa A10 y A11 — nombres originales SPSS, siempre numéricos
_educ_grupo_series  = df['A10'].apply(_educ_grupo_gse)
_ocupacion_series   = df['A11']

df['gse'] = pd.Categorical(
    _ocupacion_series.combine(
        _educ_grupo_series,
        lambda ocup, educ: np.nan if pd.isna(ocup) or educ is None
                           else _matriz_gse.get((int(ocup), educ), np.nan)
    ),
    categories=_orden_gse,
    ordered=True
)

print("GSE:", df['gse'].value_counts().reindex(_orden_gse).to_dict())

## 2. Diccionario de variables originales

In [18]:
# Tabla con nombre original y etiqueta SPSS
diccionario = pd.DataFrame({
    'variable': meta.column_names,
    'etiqueta': meta.column_labels
})
diccionario.head(20)

,variable,etiqueta
0,REGISTRO,Número de registro
1,FECHAFIN,Fecha de fin de la entrevista
2,COD_REGION,Región
3,COMUNA_DEF,Comuna
4,ZONA,ZONA
5,A9,A9.- ¿Cuál es su parentesco con el Jefe /a de ...
6,A10,"A10.- Pensando en el jefe de hogar, ¿cuál fue ..."
7,A11,A11.- Y ¿Cuál es la profesión o trabajo o acti...
8,A12_11,Aymara [A12.- ¿Algún miembro de este hogar per...
9,A12_2,Rapa-Nui [A12.- ¿Algún miembro de este hogar p...


In [19]:
# Buscar una variable por nombre o fragmento de etiqueta
busqueda = 'Q1_2'

diccionario[
    diccionario['variable'].str.contains(busqueda, case=False) |
    diccionario['etiqueta'].str.contains(busqueda, case=False, na=False)
]

,variable,etiqueta
179,Q1_2,Q1_2.- Sexo del entrevistado


In [20]:
# Ver las categorías codificadas de una variable específica
variable = 'Q1_2'

if variable in meta.variable_value_labels:
    print(f"Etiquetas de valores para '{variable}':")
    for codigo, etiqueta in meta.variable_value_labels[variable].items():
        print(f"  {codigo} -> {etiqueta}")
else:
    print(f"La variable '{variable}' no tiene etiquetas de valor registradas.")

Etiquetas de valores para 'Q1_2':
  1.0 -> Hombre
  2.0 -> Mujer


## 3. Tratamiento de NS/NR numéricos

Las variables de montos usan `9999999` como código NS/NR. Se reemplazan por `NaN`.

In [21]:
cols_nsnr = [
    'P11', 'Q7_4',
    'P17_1', 'P17_2', 'P17_3', 'P17_4', 'P17_5',
    'P19_1', 'P19_2', 'P19_3', 'P19_4',
    'Q40_1', 'Q40_2', 'Q40_3', 'Q40_4', 'Q40_5',
    'Q42', 'Q42_1'
]

for col in cols_nsnr:
    if col in df.columns:
        df[col] = df[col].replace(9999999, np.nan)

print("Valores 9999999 reemplazados por NaN.")

Valores 9999999 reemplazados por NaN.


## 4. Renombrado de variables principales

Asigna nombres cortos y legibles a las variables que se usarán en el análisis.
Las variables originales con nombre corto serán **sobreescritas** en la sección 5
con sus valores recodificados en texto.

In [22]:
def nombre_desde_etiqueta(texto, max_largo=45):
    """Convierte una etiqueta SPSS en nombre Python válido (snake_case)."""
    if not texto:
        return None
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    texto = texto.lower()
    texto = re.sub(r'\[.*?\]', '', texto)
    texto = re.sub(r'^\d+\.\-?\s*', '', texto.strip())
    texto = re.sub(r'[^a-z0-9\s]', ' ', texto)
    texto = re.sub(r'\s+', '_', texto.strip())
    return texto[:max_largo].rstrip('_')


nombres_cortos = {
    # Identificación
    'REGISTRO':      'id',
    'FECHAFIN':      'fecha_fin',
    'COD_REGION':    'region',
    'COMUNA_DEF':    'comuna',
    'ZONA':          'zona',
    # Hogar
    'A9':            'parentesco_jh',
    'A10':           'educ_jh',
    'A11':           'ocupacion_jh',
    'A12_1':         'ingreso_hogar',
    # Entrevistado
    'Q1':            'parentesco',
    'Q1_1':          'edad',
    'Q1_2':          'sexo',
    'Q1_3':          'educ',
    'Q1_4':          'ocupacion_encuestado',
    'Q2':            'actividad',
    # Acceso a internet en el hogar
    'P1':            'acceso_internet_hogar',
    'P2':            'n_smartphones_hogar',
    'P2_1':          'n_computadores_hogar',
    'P10':           'tipo_acceso_fijo',
    'P11':           'pago_mensual_internet',
    'P11_3':         'velocidad_contratada',
    'P11_4':         'calidad_acceso',
    'P11_5':         'cuota_mensual_gb',
    'P12_2':         'tipo_plan',
    'P12_1':         'plan_movil_tipo',
    'P14':           'razon_no_acceso_principal',
    'P15':           'disposicion_contratar_fijo',
    # Uso individual
    'Q5':            'uso_computador',
    'Q7':            'uso_smartphone',
    'Q7_1':          'smartphone_propio',
    'Q7_3':          'plan_movil_tipo_ind',
    'Q7_4':          'pago_mensual_movil',
    'Q9':            'ultimo_uso_internet',
    'Q10':           'frecuencia_internet',
    'Q11':           'tiempo_diario_internet',
    'Q13':           'tipo_acceso_mas_usado',
    'Q14':           'uso_internet_hogar',
    'Q15':           'frecuencia_internet_hogar',
    'Q16':           'tiempo_diario_hogar',
    'Q17':           'uso_internet_fuera_hogar',
    'Q18':           'frecuencia_fuera_hogar',
    'Q19':           'tiempo_diario_fuera_hogar',
    # Percepciones
    'Q23':           'internet_facilita_trabajo',
    'Q25':           'internet_mejora_vida',
    'Q27':           'ultima_compra_online',
    'Q31':           'percepcion_proteccion',
    'Q30_1':         'reg_control_legal',
    'Q30_2':         'reg_control_familia',
    'Q30_3':         'reg_autocontrol',
    # Factores de expansión
    'FE_HOGAR':      'fe_hogar',
    'FE_PERSONAS':   'fe_personas',
    'POND_HOGAR':    'pond_hogar',
    'POND_PERSONAS': 'pond_personas',
}

nombres_cortos_validos = {k: v for k, v in nombres_cortos.items() if k in df.columns}
df = df.rename(columns=nombres_cortos_validos)
print(f"Variables renombradas: {len(nombres_cortos_validos)}")
print(f"Columnas totales: {df.shape[1]}")

Variables renombradas: 53
Columnas totales: 587


## 5. Recodificaciones

Sobreescribe las variables renombradas con sus valores en texto.
No se crean columnas nuevas: `sexo` pasa de `1.0/2.0` a `'Hombre'/'Mujer'` directamente.

> El df resultante no tendrá columnas duplicadas ni sufijos `_rec` ni `_label`.

### 5.1 Identificación y clasificación

In [23]:
df = df.copy()

# REGIÓN
mapa_region = {
    1: 'Tarapacá', 2: 'Antofagasta', 3: 'Atacama', 4: 'Coquimbo',
    5: 'Valparaíso', 6: "O'Higgins", 7: 'Maule', 8: 'Biobío',
    9: 'Araucanía', 10: 'Los Lagos', 11: 'Aysén', 12: 'Magallanes',
    13: 'Metropolitana', 14: 'Los Ríos', 15: 'Arica y Parinacota', 16: 'Ñuble'
}
df['region'] = df['region'].map(mapa_region)

# ZONA
df['zona'] = df['zona'].map({1: 'Urbana', 2: 'Rural'})

print("Regiones en la muestra:")
print(df['region'].value_counts())

Regiones en la muestra:
region
Metropolitana         550
Valparaíso            470
Biobío                470
Maule                 430
Araucanía             430
O'Higgins             420
Los Lagos             420
Coquimbo              320
Antofagasta           300
Ñuble                 300
Los Ríos              200
Atacama               150
Tarapacá              150
Magallanes            150
Arica y Parinacota    120
Aysén                 120
Name: count, dtype: int64


### 5.2 Variables sociodemográficas del entrevistado

In [24]:
df = df.copy()

# SEXO
df['sexo'] = df['sexo'].map({1: 'Hombre', 2: 'Mujer'})

# NIVEL EDUCACIONAL
mapa_educ = {
    1: 'Sin educación formal',
    2: 'Básica incompleta',
    3: 'Básica completa',
    4: 'Media CH incompleta',
    5: 'Media TP incompleta',
    6: 'Media CH completa',
    7: 'Media TP completa',
    8: 'Superior técnica incompleta',
    9: 'Superior técnica completa',
    10: 'Superior universitaria incompleta',
    11: 'Superior universitaria completa'
}
df['educ'] = df['educ'].map(mapa_educ)

# EDUCACIÓN AGRUPADA
mapa_educ_grupo = {
    'Sin educación formal': 'Básica o menos', 'Básica incompleta': 'Básica o menos',
    'Básica completa': 'Básica o menos', 'Media CH incompleta': 'Media',
    'Media TP incompleta': 'Media', 'Media CH completa': 'Media',
    'Media TP completa': 'Media', 'Superior técnica incompleta': 'Superior',
    'Superior técnica completa': 'Superior',
    'Superior universitaria incompleta': 'Superior',
    'Superior universitaria completa': 'Superior',
}
df['educ_grupo'] = df['educ'].map(mapa_educ_grupo)

# TRAMO DE EDAD
bins   = [0, 17, 29, 44, 59, 200]
labels = ['Menor de 18', '18-29', '30-44', '45-59', '60 y más']
df['tramo_edad'] = pd.cut(df['edad'], bins=bins, labels=labels, right=True)

# ACTIVIDAD PRINCIPAL
mapa_actividad = {
    1: 'Trabajador independiente', 2: 'Empleador/patrón',
    3: 'Empleado dependiente', 4: 'Familiar no remunerado',
    5: 'FFAA y de orden', 6: 'Cesante', 7: 'Jubilado/pensionado',
    8: 'Estudiante', 9: 'Labores del hogar'
}
df['actividad'] = df['actividad'].map(mapa_actividad)

# ── GSE DERIVADO ─────────────────────────────────────────────
# Se calcula ANTES de recodificar educ_jh y ocupacion_jh,
# porque la matriz necesita los valores numéricos originales.

def _educ_grupo_gse(e):
    if pd.isna(e): return None
    e = int(e)
    if e <= 3:   return 'basica'
    elif e <= 7: return 'media'
    elif e <= 9: return 'tecnica'
    else:        return 'universitaria'

_matriz_gse = {
    (1, 'basica'): 'E',  (1, 'media'): 'E',  (1, 'tecnica'): 'D',  (1, 'universitaria'): 'D',
    (2, 'basica'): 'E',  (2, 'media'): 'D',  (2, 'tecnica'): 'D',  (2, 'universitaria'): 'C3',
    (3, 'basica'): 'D',  (3, 'media'): 'C3', (3, 'tecnica'): 'C3', (3, 'universitaria'): 'C2',
    (4, 'basica'): 'C3', (4, 'media'): 'C2', (4, 'tecnica'): 'C2', (4, 'universitaria'): 'C1',
    (5, 'basica'): 'C2', (5, 'media'): 'C1', (5, 'tecnica'): 'C1', (5, 'universitaria'): 'AB',
    (6, 'basica'): 'C1', (6, 'media'): 'AB', (6, 'tecnica'): 'AB', (6, 'universitaria'): 'AB',
}

df['_educ_grupo_jh'] = df['educ_jh'].apply(_educ_grupo_gse)

def _asignar_gse(row):
    ocup = row['ocupacion_jh']
    educ = row['_educ_grupo_jh']
    if pd.isna(ocup) or educ is None:
        return np.nan
    return _matriz_gse.get((int(ocup), educ), np.nan)

_orden_gse = ['E', 'D', 'C3', 'C2', 'C1', 'AB']
df['gse'] = pd.Categorical(
    df.apply(_asignar_gse, axis=1),
    categories=_orden_gse,
    ordered=True
)
df = df.drop(columns=['_educ_grupo_jh'])

# AHORA sí se recodifican educ_jh y ocupacion_jh a texto
mapa_educ_jh = {
    1: 'Sin educación formal', 2: 'Básica incompleta', 3: 'Básica completa',
    4: 'Media CH incompleta', 5: 'Media TP incompleta',
    6: 'Media CH completa', 7: 'Media TP completa',
    8: 'Superior técnica incompleta', 9: 'Superior técnica completa',
    10: 'Superior universitaria incompleta', 11: 'Superior universitaria completa'
}
df['educ_jh'] = df['educ_jh'].map(mapa_educ_jh)

mapa_ocupacion = {
    1: '1 - Trabajos ocasionales e informales',
    2: '2 - Oficio menor / obrero no calificado',
    3: '3 - Obrero calificado / microempresario',
    4: '4 - Empleado medio / técnico / prof. independiente',
    5: '5 - Ejecutivo medio / prof. universitario',
    6: '6 - Alto ejecutivo / empresario / directivo'
}
df['ocupacion_jh'] = df['ocupacion_jh'].map(mapa_ocupacion)
df['ocupacion_encuestado'] = df['ocupacion_encuestado'].map({
    **mapa_ocupacion,
    7: '7 - Sin trabajo remunerado'
})

print("Sexo:", df['sexo'].value_counts().to_dict())
print("GSE:", df['gse'].value_counts().reindex(_orden_gse).to_dict())

Sexo: {'Mujer': 2815, 'Hombre': 2185}
GSE: {'E': 988, 'D': 833, 'C3': 1316, 'C2': 988, 'C1': 533, 'AB': 342}


### 5.3 Acceso a Internet en el hogar

In [25]:
df = df.copy()

# ACCESO A INTERNET EN EL HOGAR
df['acceso_internet_hogar'] = df['acceso_internet_hogar'].map({1: 'Sí', 2: 'No'})

# TIPO DE ACCESO FIJO
mapa_tipo_acceso = {
    1: 'ADSL', 2: 'Cable/Módem', 3: 'Fibra óptica',
    4: 'Inalámbrica', 5: 'Satelital', 31: 'WiFi',
    32: 'Antena', 33: 'Banda ancha', 34: 'Acceso telefónico', 88: 'No sabe'
}
df['tipo_acceso_fijo'] = df['tipo_acceso_fijo'].map(mapa_tipo_acceso)

# VELOCIDAD CONTRATADA
mapa_velocidad = {
    1: 'Hasta 10 Mbps',
    2: 'Más de 10 a 100 Mbps',
    3: 'Más de 100 a 500 Mbps',
    4: 'Más de 500 Mbps a 1 Gbps',
    5: 'Más de 1 Gbps',
    99: 'NS/NR'
}
df['velocidad_contratada'] = df['velocidad_contratada'].map(mapa_velocidad)

# TIPO DE PLAN
mapa_plan = {
    1: 'Banda ancha desnuda',
    2: 'BA + TV Cable',
    3: 'BA + Telefonía fija',
    4: 'Triple pack (BA+TV+Tel)',
    5: 'Otros planes'
}
df['tipo_plan'] = df['tipo_plan'].map(mapa_plan)

print("Acceso a Internet en el hogar:")
print(df['acceso_internet_hogar'].value_counts())

Acceso a Internet en el hogar:
acceso_internet_hogar
Sí    4841
No     159
Name: count, dtype: int64


### 5.4 Uso de Internet (variables del entrevistado)

In [26]:
df = df.copy()

# USO DE COMPUTADOR Y SMARTPHONE
df['uso_computador'] = df['uso_computador'].map({1: 'Sí', 2: 'No'})
df['uso_smartphone']  = df['uso_smartphone'].map({1: 'Sí', 2: 'No'})

# ÚLTIMA VEZ QUE USÓ INTERNET
mapa_ultimo_uso = {
    1: 'Hoy',
    2: 'Entre 2 y 3 días',
    3: 'Entre 3 y 7 días',
    4: 'Entre 1 y 4 semanas',
    5: 'Más de 4 semanas',
    6: 'Más de 12 meses',
    7: 'Nunca'
}
df['ultimo_uso_internet'] = df['ultimo_uso_internet'].map(mapa_ultimo_uso)

# FRECUENCIA DE USO
mapa_frecuencia = {
    1: 'Todos los días',
    2: 'Varias veces por semana',
    3: 'Al menos una vez al mes',
    4: 'Menos de una vez al mes'
}
df['frecuencia_internet'] = df['frecuencia_internet'].map(mapa_frecuencia)

# TIEMPO DIARIO CONECTADO
mapa_tiempo = {
    1: 'Menos de 1 hora',
    2: 'Entre 1 y 2 horas',
    3: 'Entre 2 y 4 horas',
    4: 'Más de 4 horas'
}
df['tiempo_diario_internet'] = df['tiempo_diario_internet'].map(mapa_tiempo)

print("Frecuencia de uso de Internet:")
print(df['frecuencia_internet'].value_counts())

Frecuencia de uso de Internet:
frecuencia_internet
Todos los días             4199
Varias veces por semana     442
Al menos una vez al mes      59
Menos de una vez al mes      31
Name: count, dtype: int64


### 5.5 Percepciones y seguridad

In [27]:
df = df.copy()

# PERCEPCIÓN DE PROTECCIÓN
mapa_proteccion = {
    1: 'Muy protegido',
    2: 'Protegido',
    3: 'Desprotegido',
    4: 'Muy desprotegido',
    99: 'NS/NR'
}
df['percepcion_proteccion']    = df['percepcion_proteccion'].map(mapa_proteccion)
df['internet_mejora_vida']     = df['internet_mejora_vida'].map({1: 'Sí', 2: 'No'})
df['internet_facilita_trabajo']= df['internet_facilita_trabajo'].map({1: 'Sí', 2: 'No'})

print("Percepción de protección:")
print(df['percepcion_proteccion'].value_counts())

Percepción de protección:
percepcion_proteccion
Protegido           2077
Desprotegido        1653
Muy desprotegido     389
Muy protegido        381
NS/NR                231
Name: count, dtype: int64


### 5.6 GSE derivado e ingreso del hogar

El **GSE** se construye cruzando la ocupación del sostenedor principal (`ocupacion_jh` original)
con su nivel educacional (`educ_jh` original), según la metodología AIM-ESOMAR adaptada a Chile.

El **ingreso** unifica los rangos variables de `A12_1` calculando el punto medio de cada rango.

In [28]:
df = df.copy()

# ── GSE DERIVADO ─────────────────────────────────────────
# Requiere los valores numéricos originales de A10 y A11.
# educ_jh y ocupacion_jh todavía son numéricos en este punto
# porque fueron renombrados pero aún no recodificados.

def _educ_grupo(e):
    if pd.isna(e): return None
    e = int(e)
    if e <= 3:   return 'basica'
    elif e <= 7: return 'media'
    elif e <= 9: return 'tecnica'
    else:        return 'universitaria'

_matriz_gse = {
    (1, 'basica'): 'E',  (1, 'media'): 'E',  (1, 'tecnica'): 'D',  (1, 'universitaria'): 'D',
    (2, 'basica'): 'E',  (2, 'media'): 'D',  (2, 'tecnica'): 'D',  (2, 'universitaria'): 'C3',
    (3, 'basica'): 'D',  (3, 'media'): 'C3', (3, 'tecnica'): 'C3', (3, 'universitaria'): 'C2',
    (4, 'basica'): 'C3', (4, 'media'): 'C2', (4, 'tecnica'): 'C2', (4, 'universitaria'): 'C1',
    (5, 'basica'): 'C2', (5, 'media'): 'C1', (5, 'tecnica'): 'C1', (5, 'universitaria'): 'AB',
    (6, 'basica'): 'C1', (6, 'media'): 'AB', (6, 'tecnica'): 'AB', (6, 'universitaria'): 'AB',
}

df['_educ_grupo_jh'] = df['educ_jh'].apply(_educ_grupo)

def _asignar_gse(row):
    ocup = row['ocupacion_jh']
    educ = row['_educ_grupo_jh']
    if pd.isna(ocup) or educ is None:
        return np.nan
    return _matriz_gse.get((int(ocup), educ), np.nan)

_orden_gse = ['E', 'D', 'C3', 'C2', 'C1', 'AB']
df['gse'] = pd.Categorical(
    df.apply(_asignar_gse, axis=1),
    categories=_orden_gse,
    ordered=True
)
df = df.drop(columns=['_educ_grupo_jh'])

# ── INGRESO DEL HOGAR ────────────────────────────────────
_rangos = {
    11:(0,129000),   12:(130000,226000),  13:(227000,393000),  14:(394000,686000),
    15:(687000,1100000), 16:(1200000,2000000), 17:(2100000,None),
    21:(0,210000),   22:(211000,366000),  23:(367000,639000),  24:(640000,1100000),
    25:(1200000,1900000),26:(2000000,3300000),27:(3400000,None),
    31:(0,279000),   32:(280000,487000),  33:(488000,849000),  34:(850000,1400000),
    35:(1500000,2500000),36:(2600000,4500000),37:(4600000,None),
    41:(0,341000),   42:(342000,595000),  43:(596000,1000000), 44:(1100000,1800000),
    45:(1900000,3100000),46:(3200000,5500000),47:(5600000,None),
    51:(0,399000),   52:(400000,696000),  53:(697000,1200000), 54:(1300000,2100000),
    55:(2200000,3600000),56:(3700000,6400000),57:(6500000,None),
    61:(0,453000),   62:(454000,791000),  63:(792000,1300000), 64:(1400000,2400000),
    65:(2500000,4100000),66:(4200000,7300000),67:(7400000,None),
    71:(0,505000),   72:(506000,881000),  73:(882000,1500000), 74:(1600000,2600000),
    75:(2700000,4600000),76:(4700000,8100000),77:(8200000,None),
    81:(0,555000),   82:(556000,967000),  83:(968000,1600000), 84:(1700000,2900000),
    85:(3000000,5100000),86:(5200000,8900000),87:(9000000,None),
    91:(0,602000),   92:(603000,1000000), 93:(1100000,1800000),94:(1900000,3100000),
    95:(3200000,5500000),96:(5600000,9700000),97:(9800000,None),
    101:(0,648000),  102:(649000,1100000),103:(1200000,1900000),104:(2000000,3400000),
    105:(3500000,5900000),106:(6000000,10400000),107:(10500000,None),
}

def _pm(inf, sup):
    return inf * 1.5 if sup is None else (inf + sup) / 2

_mapa_pm = {float(k): _pm(v[0], v[1]) for k, v in _rangos.items()}

# ingreso_pm: valor continuo en pesos (punto medio del rango)
df['ingreso_pm'] = df['ingreso_hogar'].map(_mapa_pm)

# ingreso_tramo: tramos calibrados por percentiles del dataset
_bins_ing = [0, 384000, 540000, 798000, 1100000, 1700000, float('inf')]
_labels_ing = [
    'Hasta $384 mil', '$384 mil a $540 mil', '$540 mil a $798 mil',
    '$798 mil a $1,1 millón', '$1,1 millón a $1,7 millones', 'Más de $1,7 millones'
]
df['ingreso_tramo'] = pd.cut(df['ingreso_pm'], bins=_bins_ing, labels=_labels_ing, right=True)

# ingreso_grupo: agrupación en 3 niveles
_mapa_grupo_ing = {
    'Hasta $384 mil': 'Bajo', '$384 mil a $540 mil': 'Bajo',
    '$540 mil a $798 mil': 'Medio', '$798 mil a $1,1 millón': 'Medio',
    '$1,1 millón a $1,7 millones': 'Alto', 'Más de $1,7 millones': 'Alto',
}
df['ingreso_grupo'] = df['ingreso_tramo'].map(_mapa_grupo_ing)

print("GSE:")
print(df['gse'].value_counts().reindex(_orden_gse))

ValueError: invalid literal for int() with base 10: 'Básica completa'

In [ ]:
# Validación: ingreso promedio debe subir de E a AB
(
    df.groupby('gse', observed=True)['ingreso_pm']
    .agg(N='count', Promedio='mean', Mediana='median')
    .reindex(_orden_gse)
    .round(0)
    .astype({'N': int, 'Promedio': int, 'Mediana': int})
)

### 5.7 Recodificación de educ_jh y ocupacion_jh

Se aplica **después** de calcular el GSE, que requería los valores numéricos originales.

In [ ]:
df = df.copy()

# EDUCACIÓN DEL JEFE DE HOGAR
mapa_educ_jh = {
    1: 'Sin educación formal',
    2: 'Básica incompleta',
    3: 'Básica completa',
    4: 'Media CH incompleta',
    5: 'Media TP incompleta',
    6: 'Media CH completa',
    7: 'Media TP completa',
    8: 'Superior técnica incompleta',
    9: 'Superior técnica completa',
    10: 'Superior universitaria incompleta',
    11: 'Superior universitaria completa'
}
df['educ_jh'] = df['educ_jh'].map(mapa_educ_jh)

# OCUPACIÓN DEL JEFE DE HOGAR
df['ocupacion_jh'] = df['ocupacion_jh'].map({
    1: '1 - Trabajos ocasionales e informales',
    2: '2 - Oficio menor / obrero no calificado',
    3: '3 - Obrero calificado / microempresario',
    4: '4 - Empleado medio / técnico / prof. independiente',
    5: '5 - Ejecutivo medio / prof. universitario',
    6: '6 - Alto ejecutivo / empresario / directivo'
})

print("Educación JH:")
print(df['educ_jh'].value_counts())

## 6. Eliminación de columnas originales numéricas

Las variables renombradas que ya fueron recodificadas con texto en la sección 5
solo existen ahora con su valor textual. Las demás variables originales del SPSS
(no renombradas) se eliminan para dejar el df limpio.

> Se conservan: variables recodificadas, variables numéricas continuas útiles
> (`edad`, `pago_mensual_internet`, `ingreso_pm`, factores de expansión, etc.)
> y todas las variables de batería (`P13_X`, `Q21_X`, etc.).

In [ ]:
# Variables a conservar: las renombradas + las nuevas creadas en sección 5
vars_renombradas = set(nombres_cortos_validos.values())
vars_nuevas = {'educ_grupo', 'tramo_edad', 'gse', 'ingreso_pm', 'ingreso_tramo', 'ingreso_grupo'}

# Variables originales del SPSS (no renombradas) — se eliminan
vars_originales_spss = [
    c for c in df.columns
    if c not in vars_renombradas
    and c not in vars_nuevas
    and c == c.upper()  # las originales SPSS están en mayúsculas
]

df = df.drop(columns=vars_originales_spss)

print(f"Columnas eliminadas: {len(vars_originales_spss)}")
print(f"Columnas finales en el DataFrame: {df.shape[1]}")
print(f"\nColumnas disponibles:")
print(df.columns.tolist())

## 7. Funciones de análisis ponderado

`ORDEN_CATEGORIAS` define el orden de presentación de cada variable en las tablas.
Los nombres ya no tienen sufijos `_rec` ni `_label`.

In [ ]:
# ── Diccionario de orden categórico por variable ─────────────
ORDEN_CATEGORIAS = {
    'sexo':         ['Hombre', 'Mujer'],
    'zona':         ['Urbana', 'Rural'],
    'region': [
        'Tarapacá', 'Antofagasta', 'Atacama', 'Coquimbo', 'Valparaíso',
        "O'Higgins", 'Maule', 'Biobío', 'Araucanía', 'Los Lagos',
        'Aysén', 'Magallanes', 'Metropolitana', 'Los Ríos',
        'Arica y Parinacota', 'Ñuble'
    ],
    'educ': [
        'Sin educación formal', 'Básica incompleta', 'Básica completa',
        'Media CH incompleta', 'Media TP incompleta',
        'Media CH completa', 'Media TP completa',
        'Superior técnica incompleta', 'Superior técnica completa',
        'Superior universitaria incompleta', 'Superior universitaria completa'
    ],
    'educ_grupo':   ['Básica o menos', 'Media', 'Superior'],
    'tramo_edad':   ['Menor de 18', '18-29', '30-44', '45-59', '60 y más'],
    'actividad': [
        'Trabajador independiente', 'Empleador/patrón', 'Empleado dependiente',
        'Familiar no remunerado', 'FFAA y de orden', 'Cesante',
        'Jubilado/pensionado', 'Estudiante', 'Labores del hogar'
    ],
    'ocupacion_jh': [
        '1 - Trabajos ocasionales e informales',
        '2 - Oficio menor / obrero no calificado',
        '3 - Obrero calificado / microempresario',
        '4 - Empleado medio / técnico / prof. independiente',
        '5 - Ejecutivo medio / prof. universitario',
        '6 - Alto ejecutivo / empresario / directivo'
    ],
    'ocupacion_encuestado': [
        '1 - Trabajos ocasionales e informales',
        '2 - Oficio menor / obrero no calificado',
        '3 - Obrero calificado / microempresario',
        '4 - Empleado medio / técnico / prof. independiente',
        '5 - Ejecutivo medio / prof. universitario',
        '6 - Alto ejecutivo / empresario / directivo',
        '7 - Sin trabajo remunerado'
    ],
    'gse':              ['E', 'D', 'C3', 'C2', 'C1', 'AB'],
    'ingreso_tramo': [
        'Hasta $384 mil', '$384 mil a $540 mil', '$540 mil a $798 mil',
        '$798 mil a $1,1 millón', '$1,1 millón a $1,7 millones', 'Más de $1,7 millones'
    ],
    'ingreso_grupo':     ['Bajo', 'Medio', 'Alto'],
    'acceso_internet_hogar': ['Sí', 'No'],
    'uso_computador':        ['Sí', 'No'],
    'uso_smartphone':        ['Sí', 'No'],
    'internet_mejora_vida':  ['Sí', 'No'],
    'internet_facilita_trabajo': ['Sí', 'No'],
    'ultimo_uso_internet': [
        'Hoy', 'Entre 2 y 3 días', 'Entre 3 y 7 días',
        'Entre 1 y 4 semanas', 'Más de 4 semanas', 'Más de 12 meses', 'Nunca'
    ],
    'frecuencia_internet': [
        'Todos los días', 'Varias veces por semana',
        'Al menos una vez al mes', 'Menos de una vez al mes'
    ],
    'tiempo_diario_internet': [
        'Menos de 1 hora', 'Entre 1 y 2 horas', 'Entre 2 y 4 horas', 'Más de 4 horas'
    ],
    'percepcion_proteccion': [
        'Muy protegido', 'Protegido', 'Desprotegido', 'Muy desprotegido', 'NS/NR'
    ],
    'velocidad_contratada': [
        'Hasta 10 Mbps', 'Más de 10 a 100 Mbps', 'Más de 100 a 500 Mbps',
        'Más de 500 Mbps a 1 Gbps', 'Más de 1 Gbps', 'NS/NR'
    ],
}


# ── Función auxiliar: reordena filas según ORDEN_CATEGORIAS ──
def _ordenar_resultado(resultado_df, var, es_cruzada=False):
    if var not in ORDEN_CATEGORIAS:
        return resultado_df
    orden = ORDEN_CATEGORIAS[var]
    if es_cruzada:
        orden_valido = [v for v in orden if v in resultado_df.index]
        resto = [v for v in resultado_df.index if v not in orden_valido and v != 'Total']
        orden_final = orden_valido + resto
        if 'Total' in resultado_df.index:
            orden_final.append('Total')
        return resultado_df.reindex(orden_final)
    else:
        orden_valido = [v for v in orden if v in resultado_df[var].values]
        resto = [v for v in resultado_df[var].values if v not in orden_valido and v != 'Total']
        orden_final = orden_valido + resto + ['Total']
        resultado_df[var] = pd.Categorical(resultado_df[var], categories=orden_final, ordered=True)
        return resultado_df.sort_values(var).reset_index(drop=True)


# ── Función principal ─────────────────────────────────────────
def dstats(data_df, variables, tipo='frecuencia', variable_cruce=None,
           factor=None, transponer=False):
    """
    Realiza análisis ponderados y retorna un DataFrame.

    Parámetros:
    -----------
    data_df        : DataFrame
    variables      : str o lista de str
    tipo           : 'frecuencia', 'cruzada', 'promedio' o 'suma'
    variable_cruce : str con la variable de cruce
    factor         : nombre del factor de expansión
    transponer     : bool (solo para tipo='cruzada')

    Ejemplos:
    ---------
    dstats(df, 'sexo', tipo='frecuencia', factor='fe_personas')
    dstats(df, 'gse', tipo='cruzada', variable_cruce='zona', factor='fe_personas')
    dstats(df, ['ingreso_pm'], tipo='promedio', variable_cruce='gse', factor='fe_personas')
    """
    if not isinstance(data_df, pd.DataFrame):
        return None

    if isinstance(variables, str):
        variables = [variables]

    columnas_necesarias = variables + [factor]
    if variable_cruce:
        columnas_necesarias.append(variable_cruce)

    for col in columnas_necesarias:
        if col not in data_df.columns:
            raise ValueError(f"La columna '{col}' no existe en el DataFrame.")

    # ── FRECUENCIA ───────────────────────────────────────────
    if tipo == 'frecuencia':
        var = variables[0]
        total_pond = data_df[factor].sum()

        resultado = (
            data_df.groupby(var, observed=True)[factor]
            .sum()
            .reset_index()
            .rename(columns={factor: 'n_ponderado'})
        )
        resultado['porcentaje'] = (resultado['n_ponderado'] / total_pond * 100).round(2)

        fila_totales = pd.DataFrame({
            var: ['Total'],
            'n_ponderado': [resultado['n_ponderado'].sum()],
            'porcentaje':  [resultado['porcentaje'].sum().round(2)]
        })

        resultado = pd.concat([resultado, fila_totales], ignore_index=True)
        resultado = _ordenar_resultado(resultado, var, es_cruzada=False)
        resultado = resultado.set_index(var)
        return resultado

    # ── CRUZADA ──────────────────────────────────────────────
    elif tipo == 'cruzada':
        if not variable_cruce:
            raise ValueError("Debes especificar 'variable_cruce' para tipo='cruzada'.")
        var = variables[0]

        tabla = data_df.pivot_table(
            values=factor, index=var, columns=variable_cruce,
            aggfunc='sum', fill_value=0, observed=False
        )
        tabla_pct = tabla.div(tabla.sum(axis=0), axis=1).mul(100).round(2)

        if var in ORDEN_CATEGORIAS:
            orden_filas = [v for v in ORDEN_CATEGORIAS[var] if v in tabla.index]
            tabla     = tabla.reindex(orden_filas)
            tabla_pct = tabla_pct.reindex(orden_filas)

        if variable_cruce in ORDEN_CATEGORIAS:
            orden_cols = [v for v in ORDEN_CATEGORIAS[variable_cruce] if v in tabla.columns]
            tabla     = tabla[orden_cols]
            tabla_pct = tabla_pct[orden_cols]

        if transponer:
            tabla = tabla.T
            tabla_pct = tabla_pct.T

        tabla.loc['Total']     = tabla.sum(numeric_only=True)
        tabla_pct.loc['Total'] = tabla_pct.sum(numeric_only=True).round(2)

        columnas_intercaladas = []
        for cat in tabla.columns:
            columnas_intercaladas.append(tabla[cat].rename(f'n {cat}'))
            columnas_intercaladas.append(tabla_pct[cat].rename(f'% {cat}'))

        return pd.concat(columnas_intercaladas, axis=1)

    # ── PROMEDIO ─────────────────────────────────────────────
    elif tipo == 'promedio':
        if not variable_cruce:
            resultados = {}
            for var in variables:
                datos = data_df[[var, factor]].dropna()
                resultados[var] = float(round(np.average(datos[var], weights=datos[factor]), 4))
            return pd.DataFrame(list(resultados.items()), columns=['variable', 'promedio_ponderado'])
        else:
            filas = {}
            for grupo, subdf in data_df.groupby(variable_cruce, observed=True):
                fila = {}
                for var in variables:
                    datos = subdf[[var, factor]].dropna()
                    fila[var] = float(round(np.average(datos[var], weights=datos[factor]), 4)) \
                                if len(datos) > 0 and datos[factor].sum() > 0 else np.nan
                filas[grupo] = fila
            fila_total = {}
            for var in variables:
                datos = data_df[[var, factor]].dropna()
                fila_total[var] = float(round(np.average(datos[var], weights=datos[factor]), 4))
            filas['Total'] = fila_total
            resultado_df = pd.DataFrame(filas).T
            resultado_df.index.name = variable_cruce
            if variable_cruce in ORDEN_CATEGORIAS:
                orden_valido = [v for v in ORDEN_CATEGORIAS[variable_cruce] if v in resultado_df.index]
                resto = [v for v in resultado_df.index if v not in orden_valido and v != 'Total']
                resultado_df = resultado_df.reindex(orden_valido + resto + ['Total'])
            return resultado_df

    # ── SUMA ─────────────────────────────────────────────────
    elif tipo == 'suma':
        if not variable_cruce:
            resultados = {}
            for var in variables:
                datos = data_df[[var, factor]].dropna()
                resultados[var] = float(round((datos[var] * datos[factor]).sum(), 4))
            return pd.DataFrame(list(resultados.items()), columns=['variable', 'suma_ponderada'])
        else:
            filas = {}
            for grupo, subdf in data_df.groupby(variable_cruce, observed=True):
                fila = {}
                for var in variables:
                    datos = subdf[[var, factor]].dropna()
                    fila[var] = float(round((datos[var] * datos[factor]).sum(), 4))
                filas[grupo] = fila
            fila_total = {}
            for var in variables:
                datos = data_df[[var, factor]].dropna()
                fila_total[var] = float(round((datos[var] * datos[factor]).sum(), 4))
            filas['Total'] = fila_total
            resultado_df = pd.DataFrame(filas).T
            resultado_df.index.name = variable_cruce
            if variable_cruce in ORDEN_CATEGORIAS:
                orden_valido = [v for v in ORDEN_CATEGORIAS[variable_cruce] if v in resultado_df.index]
                resto = [v for v in resultado_df.index if v not in orden_valido and v != 'Total']
                resultado_df = resultado_df.reindex(orden_valido + resto + ['Total'])
            return resultado_df

    else:
        raise ValueError("El parámetro 'tipo' debe ser 'frecuencia', 'cruzada', 'promedio' o 'suma'.")

# **Análisis**